In [ ]:
import pathlib
from deploy.snowflake_procedure_runner import SnowflakeSProcClient
from powerflow_pipeline.util import PowerflowConfig
from dotenv import load_dotenv

from powerflow_snowflake.procedure import define_circuit, register_lfe_load_flow_udtf, run_powerflow

script_dir = pathlib.Path('lfe_install.ipynb').parent.resolve()

import os

print(f"SNOWFLAKE_ACCOUNT: {os.environ.get('SNOWFLAKE_ACCOUNT')}")
print(f"SNOWFLAKE_USER: {os.environ.get('SNOWFLAKE_USER')}")
print(f"SNOWFLAKE_ROLE: {os.environ.get('SNOWFLAKE_ROLE')}")
print(f"SNOWFLAKE_WAREHOUSE: {os.environ.get('SNOWFLAKE_WAREHOUSE')}")

In [4]:
# Build the 'src' directory so powerflow.zip includes all required modules
# (lfe_load_flow_wrapper, load_flow_engine, powerflow_snowflake, etc.)
from build_src import build_src
build_src()

Built 'src/' with: ['deploy', 'lfe_load_flow_wrapper.py', 'load_flow_engine', 'powerflow_pipeline', 'powerflow_snowflake', 'sce']


In [5]:
pf_config = PowerflowConfig()
stage_name=f"{pf_config.get_database()}.{pf_config.get_database_schema()}.{pf_config.get_stage_name()}"
print("stage_name: ", stage_name)

stage_name:  TESTDB.PUBLIC.SERVICE_STAGE


In [ ]:

load_dotenv()


def main():
    pf_config = PowerflowConfig()
    current_dir = pathlib.Path('lfe_install.ipynb').parent.resolve()
    pf_config.load_config(f"{current_dir}/env.json")
    cli = SnowflakeSProcClient(

        stage_name=f"{pf_config.get_database()}.{pf_config.get_database_schema()}.{pf_config.get_stage_name()}",
        pkg_name="powerflow",
        pkg_src="src",
        recreate_stage=True,
        local_staging_dir="SERVICE_STAGE",
    )
    procs = {
        f"{pf_config.get_database()}.{pf_config.get_database_schema()}.{pf_config.get_define_circuits_proc()}": define_circuit,
        f"{pf_config.get_database()}.{pf_config.get_database_schema()}.{pf_config.get_run_powerflow_proc()}": run_powerflow,
        f"{pf_config.get_database()}.{pf_config.get_database_schema()}.{pf_config.get_register_lfe_load_flow_udtf_proc()}": register_lfe_load_flow_udtf,
    }
    cli.install(procs)


if __name__ == "__main__":
    main()


In [ ]:
import pathlib
import pickle
from snowflake.snowpark import Session

LFE_DIR = pathlib.Path(r"c:\Users\fgonzales\git\mc_opendss\networks\lfe")
TABLE = "TESTDB.PUBLIC.NMM_F_TOPOLOGICALNODE_ENCODED_CIRCUITS_C"

connection_params = {
    "account": os.environ["SNOWFLAKE_ACCOUNT"],
    "user": os.environ["SNOWFLAKE_USER"],
    "role": os.environ["SNOWFLAKE_ROLE"],
    "warehouse": os.environ["SNOWFLAKE_WAREHOUSE"],
    "authenticator": "externalbrowser",
}

pkl_files = sorted(LFE_DIR.glob("*.pkl"))
print(f"Found {len(pkl_files)} pickle files in {LFE_DIR}")

with Session.builder.configs(connection_params).create() as session:
    cur = session.connection.cursor()
    inserted = 0
    failed = 0
    for pkl_path in pkl_files:
        # Extract CIRCUIT_KEY from filename: CKT_XXXX_XXXXX_lfe.pkl -> CKT_XXXX_XXXXX
        stem = pkl_path.stem  # e.g. "CKT_4799_01085_lfe"
        circuit_key = stem.replace("_lfe", "")

        pkl_bytes = pkl_path.read_bytes()
        hex_str = pkl_bytes.hex()

        # Try to extract FEEDER_MRID from the pickle object
        feeder_mrid = None
        try:
            net = pickle.loads(pkl_bytes)
            if hasattr(net, "FEEDER_MRID"):
                feeder_mrid = net.FEEDER_MRID
            elif hasattr(net, "feeder_mrid"):
                feeder_mrid = net.feeder_mrid
        except Exception:
            pass

        feeder_val = f"'{feeder_mrid}'" if feeder_mrid else "''"

        try:
            cur.execute(
                f"INSERT INTO {TABLE} (CIRCUIT_KEY, STATUS, FEEDER_MRID, PRELOAD_ENCODED_CA) "
                f"VALUES ('{circuit_key}', 'SUCCESS', {feeder_val}, TO_BINARY('{hex_str}', 'HEX'))"
            )
            inserted += 1
            if inserted % 50 == 0:
                print(f"  Inserted {inserted} so far...")
        except Exception as e:
            failed += 1
            print(f"FAILED {circuit_key}: {e}")

    print(f"Done. Inserted: {inserted}, Failed: {failed}")